# 009 GNN + NTL/Proximity Prior-Loss Training

**Prerequisite**: 006 uses NTL/Proximity post-processing to correct GNN predictions. This notebook integrates NTL and Proximity into the training loop,
using a Forward KL-divergence prior loss to directly constrain the shape of `w_{s→a}`, thereby eliminating the post-processing step.

**Pipeline**: load data → precompute NTL/Proximity/RCI → inject into HeteroData → train (L_landuse + L_ntl + L_prox) → predict → aggregate

| Aggregation path | Description |
|----------|------|
| voronoi_GNN | Voronoi nearest-neighbor allocation + GNN-weighted demand (no post-processing) |
| civd_GNN | HDBSCAN cluster allocation + GNN-weighted demand (no post-processing) |

**Key design choices**:
- Forward KL(target ‖ w): mode-covering, w must cover the high-value regions of target
- RCI filtering: only RCI agents (lu_res + lu_com + lu_ind > 0.5) participate in target
- log(1+x) transform: compresses the right-skewed distribution of raw values, aligning with the post-hoc correction

In [ ]:
import sys
import pickle
import time
import json
from datetime import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import torch
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler

# Project root directory
PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from SpatialAllocation.GNN.utils.GraphBuilder import (
    preprocess_features, prepare_hetero_graph_from_processed
)
from SpatialAllocation.GNN.core.EdgeWeightSolver import EdgeWeightSolver
from SpatialAllocation.GNN.core.ModelConfig import ModelConfig
from SpatialAllocation.Allocator import allocator_registry
from SpatialAllocation.Allocator.clustering.do_clustering import do_clustering
from SpatialAllocation.Weighter import weighter_registry
from SpatialAllocation.FeatureExtractor.correctors.proximity_corrector import ProximityCorrector

warnings.filterwarnings('ignore', category=FutureWarning)

# ====== Path configuration ======
DATA_DIR = Path('./results/intermediate')
ASSEMBLED_DIR = DATA_DIR / 'features' / 'assembled'
EXTRACTED_DIR = DATA_DIR / 'features' / 'extracted'
OUTPUT_DIR = Path('./results/static_allocation')

# ====== Experiment name ======
EXPERIMENT_NAME = 'GNN_ntl_prox'
GNN_DIR = DATA_DIR / 'GNN_TEST' / EXPERIMENT_NAME
GNN_DIR.mkdir(parents=True, exist_ok=True)

# Leaderboard lives at the GNN_TEST root
LEADERBOARD_PATH = DATA_DIR / 'GNN_TEST' / 'gnn_leaderboard.json'

# ====== Training configuration ======
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

OVERWRITE = True
MODEL_NAME = f'model_{SEED}_ntl_prox_prior.pth'
MODEL_PATH = GNN_DIR / MODEL_NAME

# ====== Region configuration ======
TRAIN_LOCATIONS = [
    'London',
    'TLH2', 'TLH3', 'TLJ1', 'TLF1', 'TLF2',
    'TLC1', 'TLC2', 'TLD6', 'TLG1', 'TLG2', 'TLE4',
]
TEST_LOCATIONS = ['TLH1', 'TLE3', 'TLD3', 'TLD4']
STUDY_REGIONS = TRAIN_LOCATIONS + TEST_LOCATIONS

# ====== Agent connectivity type ======
AGENT_CONNECTIVITY = 'moore'

# ====== Feature list ======
AGENT_FEATURE_COLS = [
    'lu_residential_prop', 'lu_commercial_prop', 'lu_industrial_prop',
    'lu_agricultural_prop', 'lu_others_prop',
]

SOURCE_FEATURE_COLS = [
    'residential_percent', 'commercial_percent', 'industrial_percent',
    'agricultural_percent', 'others_percent',
]

# ====== RCI threshold ======
RCI_THRESHOLD = 0.5

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Agent features: {len(AGENT_FEATURE_COLS)}-dim')
print(f'Source features: {len(SOURCE_FEATURE_COLS)}-dim')
print(f'Model path: {MODEL_PATH}')
print(f'OVERWRITE: {OVERWRITE}')
print(f'Train regions: {len(TRAIN_LOCATIONS)}')
print(f'Test regions: {len(TEST_LOCATIONS)}')
print(f'Agent connectivity: {AGENT_CONNECTIVITY}')

In [ ]:
# ─── Load data + precompute NTL / Proximity / RCI ───
region_gdf = gpd.read_file(str(DATA_DIR / 'ITL3_region.gpkg'))
substations_gdf = gpd.read_file(str(DATA_DIR / 'substations.gpkg'))

print(f'ITL3 regions: {region_gdf.shape[0]} rows')
print(f'Substations: {substations_gdf.shape[0]} rows')

# Load assembled grid_gdf
grids = {}
for loc in STUDY_REGIONS:
    path = ASSEMBLED_DIR / f'{loc}_grid_points.pickle'
    with open(path, 'rb') as f:
        grid_gdf, step_size_m = pickle.load(f)
    grids[loc] = (grid_gdf, step_size_m)
    print(f'  {loc}: {len(grid_gdf)} grid points, step={step_size_m}m')

# Load NTL data
ntl_dict = {}
for loc in STUDY_REGIONS:
    ntl_path = EXTRACTED_DIR / f'{loc}_ntl.npz'
    ntl_npz = np.load(ntl_path, allow_pickle=True)
    ntl_values = ntl_npz['data'][:, 0]
    ntl_dict[loc] = ntl_values
    grid_gdf, _ = grids[loc]
    assert len(ntl_values) == len(grid_gdf), \
        f'{loc}: NTL length {len(ntl_values)} != grid length {len(grid_gdf)}'
    print(f'  {loc} NTL: N={len(ntl_values)}, range=[{ntl_values.min():.2f}, {ntl_values.max():.2f}]')

# Organize data dicts by region
region_dict = {}
subs_dict = {}

for loc in STUDY_REGIONS:
    grid_gdf, step_size_m = grids[loc]
    study_itl3 = grid_gdf['ITL3'].unique()
    region_dict[loc] = region_gdf[region_gdf['ITL3'].isin(study_itl3)].copy()
    subs_dict[loc] = substations_gdf[substations_gdf['ITL3'].isin(study_itl3)].copy().reset_index(drop=True)
    print(f'  {loc}: {len(region_dict[loc])} ITL3 regions, {len(subs_dict[loc])} substations')

# ── Precompute Proximity scores ──
proximity_dict = {}
for loc in STUDY_REGIONS:
    grid_gdf, _ = grids[loc]
    subs_sub = subs_dict[loc]
    prox_scores = ProximityCorrector.compute_scores(grid_gdf, subs_sub, gamma=1.0)
    proximity_dict[loc] = prox_scores
    print(f'  {loc} Proximity: range=[{prox_scores.min():.4f}, {prox_scores.max():.4f}]')

# ── Precompute RCI mask ──
rci_dict = {}
for loc in STUDY_REGIONS:
    grid_gdf, _ = grids[loc]
    rci = (grid_gdf['lu_residential_prop'] + grid_gdf['lu_commercial_prop']
           + grid_gdf['lu_industrial_prop']).values
    rci_dict[loc] = rci > RCI_THRESHOLD
    print(f'  {loc} RCI: {rci_dict[loc].sum()}/{len(rci_dict[loc])} ({rci_dict[loc].mean():.1%})')

# Demand apportionment
LANDUSE_PERCENT_MAP = {
    'lu_residential_prop': 'residential_percent',
    'lu_commercial_prop': 'commercial_percent',
    'lu_industrial_prop': 'industrial_percent',
    'lu_agricultural_prop': 'agricultural_percent',
    'lu_others_prop': 'others_percent',
}
LU_COLS = list(LANDUSE_PERCENT_MAP.keys())
PCT_COLS = list(LANDUSE_PERCENT_MAP.values())


def compute_demand(grid_gdf, region_sub, weighter_result, demand_col='demand'):
    """Convert the weighter's output weights + region percentages into grid demand."""
    W = weighter_result.weights
    gdf = grid_gdf.copy()
    gdf[demand_col] = 0.0
    region_info = region_sub.set_index('ITL3')

    for itl3, group in gdf.groupby('ITL3'):
        if itl3 not in region_info.index:
            continue
        total_demand = region_info.loc[itl3, 'Demand (MVA)']
        idx = group.index

        if W.ndim == 2:
            pcts = np.array([region_info.loc[itl3, c] for c in PCT_COLS])
            score = W[idx] @ pcts
        else:
            score = W[idx]

        score_sum = score.sum()
        if score_sum > 0:
            gdf.loc[idx, demand_col] = total_demand * score / score_sum
        else:
            gdf.loc[idx, demand_col] = total_demand / len(group)

    return gdf


# Compute landuse_demand for each region
for loc in STUDY_REGIONS:
    grid_gdf, step_size_m = grids[loc]
    region_sub = region_dict[loc]
    subs_sub = subs_dict[loc]

    gpm = weighter_registry.create('gpm', config={'mode': 'categorical', 'proportion_columns': LU_COLS})
    gpm_res = gpm.compute(grid_gdf, target_gdf=subs_sub)
    grid_gdf = compute_demand(grid_gdf, region_sub, gpm_res, demand_col='landuse_demand')
    grids[loc] = (grid_gdf, step_size_m)

    print(f'  {loc} landuse_demand total: {grid_gdf["landuse_demand"].sum():.1f} MVA')

In [ ]:
# ─── Build the HeteroData graph + inject NTL/Proximity/RCI ───
graphs = {}

LU_PROP_TO_CATEGORY = {
    'lu_residential_prop': 'residential',
    'lu_commercial_prop': 'commercial',
    'lu_industrial_prop': 'industrial',
    'lu_agricultural_prop': 'agricultural',
    'lu_others_prop': 'others',
}

for loc in STUDY_REGIONS:
    grid_gdf, step_size_m = grids[loc]
    region_sub = region_dict[loc]
    subs_sub = subs_dict[loc]

    # Derive the landuse categorical column
    lu_prop_cols = [c for c in LU_PROP_TO_CATEGORY if c in grid_gdf.columns]
    if 'landuse' not in grid_gdf.columns and lu_prop_cols:
        categories = [LU_PROP_TO_CATEGORY[c] for c in lu_prop_cols]
        max_idx = grid_gdf[lu_prop_cols].values.argmax(axis=1)
        grid_gdf = grid_gdf.copy()
        grid_gdf['landuse'] = [categories[i] for i in max_idx]
        grids[loc] = (grid_gdf, step_size_m)

    # Coordinate projection + normalization
    gdf_a = grid_gdf.copy().to_crs('EPSG:3857')
    gdf_t = subs_sub.copy().to_crs('EPSG:3857')
    gdf_s = region_sub.copy()
    gdf_s['geometry'] = gdf_s.geometry.centroid.to_crs('EPSG:3857')

    coords_a = np.column_stack([gdf_a.geometry.x, gdf_a.geometry.y])
    coords_t = np.column_stack([gdf_t.geometry.x, gdf_t.geometry.y])
    coords_s = np.column_stack([gdf_s.geometry.x, gdf_s.geometry.y])

    scaler = StandardScaler().fit(np.vstack([coords_a, coords_t, coords_s]))

    from shapely.geometry import Point

    coords_a_scaled = scaler.transform(coords_a)
    coords_s_scaled = scaler.transform(coords_s)

    gdf_a_scaled = gdf_a.copy()
    gdf_a_scaled['geometry'] = [Point(x, y) for x, y in coords_a_scaled]

    gdf_s_scaled = gdf_s.copy()
    gdf_s_scaled['geometry'] = [Point(x, y) for x, y in coords_s_scaled]

    agent_cols_for_graph = [c for c in AGENT_FEATURE_COLS if c in gdf_a_scaled.columns]
    source_cols_for_graph = [c for c in SOURCE_FEATURE_COLS if c in gdf_s_scaled.columns]

    features_a = preprocess_features(gdf_a_scaled[agent_cols_for_graph + ['geometry']])
    features_s = preprocess_features(gdf_s_scaled[source_cols_for_graph + ['geometry']])

    print(f'{loc} agent feature dim: {features_a["final_features"].shape}')
    print(f'{loc} source feature dim: {features_s["final_features"].shape}')

    hetero_data = prepare_hetero_graph_from_processed(
        gdf_s_scaled, gdf_a_scaled,
        processed_features_s=features_s,
        processed_features_a=features_a,
        relation_column='ITL3',
        agent_connectivity=AGENT_CONNECTIVITY,
    )

    # ── Inject NTL / Proximity / RCI into HeteroData ──
    hetero_data['agent'].ntl_values = torch.tensor(ntl_dict[loc], dtype=torch.float32)
    hetero_data['agent'].proximity_scores = torch.tensor(proximity_dict[loc], dtype=torch.float32)
    hetero_data['agent'].rci_mask = torch.tensor(rci_dict[loc], dtype=torch.bool)

    graphs[loc] = hetero_data
    torch.save(hetero_data, GNN_DIR / f'{loc}_graph.pt')

    # Validation
    split = 'TRAIN' if loc in TRAIN_LOCATIONS else 'TEST'
    print(f'\n{loc} [{split}] HeteroData:')
    print(f'  source nodes: {hetero_data["source"].x.shape}')
    print(f'  agent nodes: {hetero_data["agent"].x.shape}')
    print(f'  source→agent edges: {hetero_data["source", "connects_to", "agent"].edge_index.shape}')
    if ('agent', 'near', 'agent') in hetero_data.edge_types:
        print(f'  agent↔agent edges: {hetero_data["agent", "near", "agent"].edge_index.shape}')
    has_landuse = hasattr(hetero_data, 'landuse_mapping_matrix')
    print(f'  landuse supervision matrix: {"yes" if has_landuse else "no"}')
    print(f'  NTL: range=[{hetero_data["agent"].ntl_values.min():.2f}, {hetero_data["agent"].ntl_values.max():.2f}]')
    print(f'  Proximity: range=[{hetero_data["agent"].proximity_scores.min():.4f}, {hetero_data["agent"].proximity_scores.max():.4f}]')
    print(f'  RCI: {hetero_data["agent"].rci_mask.sum()}/{hetero_data["agent"].rci_mask.shape[0]}')

In [ ]:
# ====== Train the GNN (L_landuse + L_ntl_prior + L_proximity_prior) ======
config = ModelConfig(
    epochs=200,
    hidden_dim=256,
    embedding_dim=128,
    num_layers=3,
    conv_type='hgt',
    allocation_temperature_start=0.01,
    learning_rate=1e-3,
    weight_decay=1e-4,
    use_scheduler=True,
    warmup_epochs=20,
    decay_epochs=20,
    cosine_epochs=160,
    cosine_eta_min=1e-5,
    learnable=False,
    save_path=str(MODEL_PATH),
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

train_graphs = [graphs[loc] for loc in TRAIN_LOCATIONS]
test_graphs = [graphs[loc] for loc in TEST_LOCATIONS]
train_dl = DataLoader(train_graphs, batch_size=1, shuffle=False)
test_dl = DataLoader(test_graphs, batch_size=1, shuffle=False)

solver = EdgeWeightSolver(config)
objective_weights = {
    'landuse_prediction_loss': 1.0,
    'ntl_prior': 0.1,
    'proximity_prior': 0.1,
}

if not OVERWRITE and MODEL_PATH.exists():
    solver.init_model(train_dl, objective_weights)
    print(f'Existing model detected: {MODEL_PATH.name}, skipping training. Set OVERWRITE=True to force retraining.')
else:
    print(f'Training config: {config.conv_type}, hidden={config.hidden_dim}, epochs={config.epochs}')
    print(f'Device: {config.device}')
    print(f'Loss function: {objective_weights}')
    print(f'Number of training graphs: {len(train_graphs)}')
    print(f'Number of test graphs: {len(test_graphs)}')
    print()

    solver.train_multi_graph(train_dl, test_dataloader=test_dl, objective_weights=objective_weights)

In [ ]:
# ─── Predict edge weights → aggregate demand to substations (no post-processing) ───
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error

results_gnn = {}

for loc in STUDY_REGIONS:
    grid_gdf, step_size_m = grids[loc]
    region_sub = region_dict[loc]
    subs_sub = subs_dict[loc]
    graph = graphs[loc]

    # Predict source→agent edge weights
    edge_weights_df = solver.predict_edge_weights(graph)
    print(f'{loc}: predicted {len(edge_weights_df)} edge weights')

    # Verify per-source normalization
    weight_sums = edge_weights_df.groupby('source_node_idx')['predicted_weight'].sum()
    print(f'  per-source weight sum: mean={weight_sums.mean():.4f}, std={weight_sums.std():.6f}')

    # Map edge weights back to grid_gdf → gnn_demand
    region_info = region_sub.set_index('ITL3')
    source_index_map = graph.source_index_map

    grid_gdf = grid_gdf.copy()
    grid_gdf['gnn_demand'] = 0.0

    for _, row in edge_weights_df.iterrows():
        s_idx = int(row['source_node_idx'])
        a_orig_idx = int(row['agent_original_idx'])
        w = row['predicted_weight']

        s_orig_idx = source_index_map.iloc[s_idx]
        itl3 = region_sub.loc[s_orig_idx, 'ITL3']
        total_demand = region_info.loc[itl3, 'Demand (MVA)']

        grid_gdf.loc[a_orig_idx, 'gnn_demand'] += w * total_demand

    print(f'  GNN demand total: {grid_gdf["gnn_demand"].sum():.1f} MVA')
    print(f'  Region demand total: {region_sub["Demand (MVA)"].sum():.1f} MVA')

    grids[loc] = (grid_gdf, step_size_m)

    # ── Voronoi allocation ──
    alloc_voronoi = allocator_registry.create('voronoi')
    voronoi_res = alloc_voronoi.allocate(grid_gdf, subs_sub)

    demands = grid_gdf['gnn_demand'].values

    # Path 1: voronoi_GNN
    subs_result_voronoi = subs_sub.copy()
    subs_result_voronoi['allocated_demand'] = 0.0
    for target_idx in range(len(subs_sub)):
        mask = voronoi_res.assignment == target_idx
        subs_result_voronoi.loc[target_idx, 'allocated_demand'] = demands[mask].sum()
    print(f'  voronoi_GNN allocation total: {subs_result_voronoi["allocated_demand"].sum():.1f} MVA')

    # ── CIVD allocation ──
    coords_4326 = np.column_stack([subs_sub.geometry.x.values, subs_sub.geometry.y.values])
    cluster_gdf, centroid_gdf = do_clustering(coords_4326, method='hdbscan', min_cluster_size=2)
    centroid_gdf = centroid_gdf.reset_index(drop=True)
    print(f'  HDBSCAN cluster count: {cluster_gdf["cluster_label"].nunique()}')

    target_civd = subs_sub.copy()
    target_civd['cluster_label'] = cluster_gdf['cluster_label']

    civd_config = {
        'solver': 'scip',
        'method': 'civd',
        'cluster_label_column': 'cluster_label',
        'n_jobs': -1,
    }

    # Reuse cache
    civd_cache_path = OUTPUT_DIR / f'{loc}_civd_cache.pickle'
    if civd_cache_path.exists():
        with open(civd_cache_path, 'rb') as f:
            cache = pickle.load(f)
        civd_assignment = cache['assignment']
        print(f'  Reusing CIVD cache: {civd_cache_path.name}')
    else:
        alloc_civd = allocator_registry.create('civd', config=civd_config)
        civd_res = alloc_civd.allocate(grid_gdf, target_civd)
        civd_assignment = civd_res.assignment
        print(f'  CIVD solve complete')

    # Path 2: civd_GNN
    subs_result_civd = subs_sub.copy()
    subs_result_civd['allocated_demand'] = 0.0
    cluster_demands = {}
    for label in np.unique(civd_assignment):
        mask = civd_assignment == label
        cluster_demands[label] = demands[mask].sum()
    for label, total_d in cluster_demands.items():
        members = cluster_gdf[cluster_gdf['cluster_label'] == label].index
        n_members = len(members)
        if n_members > 0:
            for idx in members:
                if idx < len(subs_result_civd):
                    subs_result_civd.loc[idx, 'allocated_demand'] += total_d / n_members
    print(f'  civd_GNN allocation total: {subs_result_civd["allocated_demand"].sum():.1f} MVA')

    results_gnn[loc] = {
        'voronoi_GNN': subs_result_voronoi,
        'civd_GNN': subs_result_civd,
    }

In [ ]:
# ─── Compute RMSE / Corr / MAE ───

def evaluate_allocation(subs_result, actual_col='Demand (MVA)', alloc_col='allocated_demand'):
    actual = subs_result[actual_col].values
    allocated = subs_result[alloc_col].values
    corr, _ = pearsonr(actual, allocated)
    rmse = np.sqrt(mean_squared_error(actual, allocated))
    mae = mean_absolute_error(actual, allocated)
    conservation_error = abs(allocated.sum() - actual.sum()) / actual.sum()
    return {
        'corr': corr, 'rmse': rmse, 'mae': mae,
        'conservation_error': conservation_error,
        'total_allocated': allocated.sum(),
        'total_actual': actual.sum(),
    }


# Load 003 baseline RMSE
rmse_csv = OUTPUT_DIR / 'all_regions_rmse.csv'
rmse_table = pd.read_csv(rmse_csv, index_col=0)

all_metrics = []
for loc in STUDY_REGIONS:
    split = 'TRAIN' if loc in TRAIN_LOCATIONS else 'TEST'
    for method_name, subs_result in results_gnn[loc].items():
        m = evaluate_allocation(subs_result)
        rmse_table.loc[method_name, loc] = round(m['rmse'], 4)
        all_metrics.append({'region': loc, 'split': split, 'method': method_name, **m})
        print(f'[{split}] {loc} {method_name}: corr={m["corr"]:.4f}, RMSE={m["rmse"]:.4f}, MAE={m["mae"]:.4f}')

gnn_rmse_csv = GNN_DIR / 'all_regions_rmse.csv'
rmse_table.to_csv(gnn_rmse_csv)
print(f'\nRMSE table saved to: {gnn_rmse_csv}')

print(f'\n=== Updated RMSE Table ===')
display(rmse_table)

gnn_metrics_df = pd.DataFrame(all_metrics).round(4)
print('\n=== GNN Method Detailed Metrics ===')
display(gnn_metrics_df[['split', 'region', 'method', 'corr', 'rmse', 'mae', 'conservation_error']])

print('\n=== Train vs Test Average Metrics ===')
summary = gnn_metrics_df.groupby(['split', 'method'])[['corr', 'rmse', 'mae']].mean().round(4)
display(summary)

In [ ]:
# ─── Cross-Region Correlation / MAE Summary ───

corr_csv = OUTPUT_DIR / 'all_regions_corr.csv'
mae_csv_path = OUTPUT_DIR / 'all_regions_mae.csv'

corr_table = pd.read_csv(corr_csv, index_col=0) if corr_csv.exists() else pd.DataFrame()
mae_table_base = pd.read_csv(mae_csv_path, index_col=0) if mae_csv_path.exists() else pd.DataFrame()

for loc in STUDY_REGIONS:
    for method_name, subs_result in results_gnn[loc].items():
        m = evaluate_allocation(subs_result)
        corr_table.loc[method_name, loc] = round(m['corr'], 4)
        mae_table_base.loc[method_name, loc] = round(m['mae'], 4)

gnn_corr_csv = GNN_DIR / 'all_regions_corr.csv'
corr_table.to_csv(gnn_corr_csv)
print(f'Correlation table saved to: {gnn_corr_csv}')

print(f'\n=== Updated Correlation Table ===')
display(corr_table.round(4))

gnn_mae_csv = GNN_DIR / 'all_regions_mae.csv'
mae_table_base.to_csv(gnn_mae_csv)
print(f'\nMAE table saved to: {gnn_mae_csv}')
print(f'\n=== Updated MAE Table ===')
display(mae_table_base.round(4))

In [ ]:
# ─── Visualization: GNN scatter plots + RMSE bar chart ───
loc = STUDY_REGIONS[0]
subs_sub = subs_dict[loc]
actual = subs_sub['Demand (MVA)'].values

# Load the best 003 baseline
baseline_path = OUTPUT_DIR / f'{loc}_all_results.pickle'
if baseline_path.exists():
    with open(baseline_path, 'rb') as f:
        baseline_data = pickle.load(f)
    best_baseline_name = 'ITL3_average'
    best_baseline = baseline_data['results'][best_baseline_name]
else:
    best_baseline_name = None

# ── Scatter plots ──
loc_results = results_gnn[loc]
gnn_methods = list(loc_results.keys())
n_plots = len(gnn_methods) + (1 if best_baseline_name else 0)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

plot_idx = 0

if best_baseline_name:
    ax = axes[plot_idx]
    allocated = best_baseline['allocated_demand'].values
    ax.scatter(actual, allocated, alpha=0.3, s=10, color='gray')
    max_val = max(actual.max(), allocated.max()) * 1.1
    ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5)
    rmse_val = rmse_table.loc[best_baseline_name, loc]
    ax.set_title(f'{best_baseline_name}\nRMSE={rmse_val:.2f}')
    ax.set_xlabel('Actual Demand (MVA)')
    ax.set_ylabel('Allocated Demand (MVA)')
    plot_idx += 1

for method_name in gnn_methods:
    ax = axes[plot_idx]
    allocated = loc_results[method_name]['allocated_demand'].values
    m = evaluate_allocation(loc_results[method_name])
    ax.scatter(actual, allocated, alpha=0.3, s=10, color='steelblue')
    max_val = max(actual.max(), allocated.max()) * 1.1
    ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5)
    ax.set_title(f'{method_name}\ncorr={m["corr"]:.3f}, RMSE={m["rmse"]:.2f}')
    ax.set_xlabel('Actual Demand (MVA)')
    ax.set_ylabel('Allocated Demand (MVA)')
    plot_idx += 1

plt.suptitle(f'{loc} - NTL+Prox Prior GNN vs baseline (N={len(subs_sub)})', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(GNN_DIR / f'{loc}_gnn_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# ── RMSE bar chart ──
fig, ax = plt.subplots(figsize=(8, 5))
methods = list(rmse_table.index)
rmse_vals = rmse_table[loc].values.astype(float)

colors = ['steelblue' if 'GNN' in m else 'lightgray' for m in methods]
bars = ax.barh(methods, rmse_vals, color=colors, edgecolor='gray')
ax.set_xlabel('RMSE (MVA)')
ax.set_title(f'{loc} - RMSE Comparison Across All Methods')
ax.invert_yaxis()

for bar, val in zip(bars, rmse_vals):
    ax.text(val + 0.2, bar.get_y() + bar.get_height() / 2, f'{val:.2f}',
            va='center', fontsize=9)

plt.tight_layout()
plt.savefig(GNN_DIR / f'{loc}_gnn_rmse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nCharts saved to {GNN_DIR}')

In [ ]:
# ─── Leaderboard: RRMSE comparison ───

BASELINE_METHOD = 'ITL3_average'
GNN_METHOD = 'civd_GNN'

# ── 1. Compute RRMSE for the current experiment ──
per_region_rrmse = {}
per_region_corr = {}
for loc in STUDY_REGIONS:
    baseline_rmse = float(rmse_table.loc[BASELINE_METHOD, loc])
    gnn_rmse = float(rmse_table.loc[GNN_METHOD, loc])
    per_region_rrmse[loc] = round(gnn_rmse / baseline_rmse, 4) if baseline_rmse > 0 else None

    gnn_corr_val = corr_table.loc[GNN_METHOD, loc]
    per_region_corr[loc] = round(float(gnn_corr_val), 4) if pd.notna(gnn_corr_val) else None

train_rrmse_vals = [per_region_rrmse[loc] for loc in TRAIN_LOCATIONS if per_region_rrmse[loc] is not None]
test_rrmse_vals = [per_region_rrmse[loc] for loc in TEST_LOCATIONS if per_region_rrmse[loc] is not None]
train_corr_vals = [per_region_corr[loc] for loc in TRAIN_LOCATIONS if per_region_corr[loc] is not None]
test_corr_vals = [per_region_corr[loc] for loc in TEST_LOCATIONS if per_region_corr[loc] is not None]

current_entry = {
    'experiment_name': EXPERIMENT_NAME,
    'timestamp': datetime.now().isoformat(),
    'test_median_rrmse': round(float(np.median(test_rrmse_vals)), 4),
    'train_median_rrmse': round(float(np.median(train_rrmse_vals)), 4),
    'test_median_corr': round(float(np.median(test_corr_vals)), 4),
    'train_median_corr': round(float(np.median(train_corr_vals)), 4),
    'per_region_rrmse': per_region_rrmse,
    'per_region_corr': per_region_corr,
    'config_snapshot': {
        'conv_type': config.conv_type,
        'hidden_dim': config.hidden_dim,
        'embedding_dim': config.embedding_dim,
        'epochs': config.epochs,
        'learning_rate': config.learning_rate,
        'objective_weights': objective_weights,
    },
}

# ── 2. Load the leaderboard ──
if LEADERBOARD_PATH.exists():
    with open(LEADERBOARD_PATH, 'r', encoding='utf-8') as f:
        leaderboard = json.load(f)
else:
    leaderboard = {
        'version': 1,
        'baseline_method': BASELINE_METHOD,
        'gnn_method': GNN_METHOD,
        'global_best': None,
        'history': [],
    }

# ── 3. Compare and update ──
prev_best = leaderboard['global_best']
is_best = (prev_best is None) or (current_entry['test_median_rrmse'] < prev_best['test_median_rrmse'])

if is_best:
    leaderboard['global_best'] = {k: v for k, v in current_entry.items()}

history_entry = {
    'experiment_name': EXPERIMENT_NAME,
    'timestamp': current_entry['timestamp'],
    'test_median_rrmse': current_entry['test_median_rrmse'],
    'train_median_rrmse': current_entry['train_median_rrmse'],
    'test_median_corr': current_entry['test_median_corr'],
    'train_median_corr': current_entry['train_median_corr'],
    'is_best': is_best,
}
leaderboard['history'].append(history_entry)

# ── 4. Save ──
with open(LEADERBOARD_PATH, 'w', encoding='utf-8') as f:
    json.dump(leaderboard, f, indent=2, ensure_ascii=False)

# ── 5. Print results ──
print(f'Experiment: {EXPERIMENT_NAME}')
print(f'Leaderboard: {LEADERBOARD_PATH}')
print()

rrmse_rows = []
for loc in STUDY_REGIONS:
    split = 'TRAIN' if loc in TRAIN_LOCATIONS else 'TEST'
    rrmse_val = per_region_rrmse[loc]
    corr_val = per_region_corr[loc]
    baseline_rmse = float(rmse_table.loc[BASELINE_METHOD, loc])
    gnn_rmse = float(rmse_table.loc[GNN_METHOD, loc])
    status = '< baseline' if rrmse_val < 1.0 else '> baseline'
    rrmse_rows.append({
        'region': loc, 'split': split,
        f'RMSE({BASELINE_METHOD})': round(baseline_rmse, 2),
        f'RMSE({GNN_METHOD})': round(gnn_rmse, 2),
        'RRMSE': rrmse_val, 'Corr': corr_val, 'status': status,
    })

rrmse_df = pd.DataFrame(rrmse_rows)
print('=== Per-Region RRMSE ===')
display(rrmse_df)

print(f'\n=== Current Experiment Summary ===')
print(f'  Test  median RRMSE: {current_entry["test_median_rrmse"]:.4f}')
print(f'  Train median RRMSE: {current_entry["train_median_rrmse"]:.4f}')
print(f'  Test  median Corr:  {current_entry["test_median_corr"]:.4f}')
print(f'  Train median Corr:  {current_entry["train_median_corr"]:.4f}')

best = leaderboard['global_best']
if is_best:
    if prev_best is None:
        print(f'\n*** First record, set as global best ***')
    else:
        delta = prev_best['test_median_rrmse'] - current_entry['test_median_rrmse']
        print(f'\n*** New global best! (RRMSE reduced by {delta:.4f}, previous best: {prev_best["experiment_name"]}) ***')
else:
    delta = current_entry['test_median_rrmse'] - best['test_median_rrmse']
    print(f'\nDid not surpass the global best "{best["experiment_name"]}" (RRMSE gap +{delta:.4f})')
    print(f'  Best test_median_rrmse: {best["test_median_rrmse"]:.4f}')

    wins, losses = [], []
    for loc in TEST_LOCATIONS:
        cur = per_region_rrmse[loc]
        bst = best['per_region_rrmse'].get(loc)
        if cur is not None and bst is not None:
            if cur < bst:
                wins.append(f'{loc}({cur:.3f} vs {bst:.3f})')
            else:
                losses.append(f'{loc}({cur:.3f} vs {bst:.3f})')
    if wins:
        print(f'  Winning regions: {", ".join(wins)}')
    if losses:
        print(f'  Lagging regions: {", ".join(losses)}')

print(f'\n=== Leaderboard History ({len(leaderboard["history"])} entries) ===')
hist_df = pd.DataFrame(leaderboard['history'])
hist_df = hist_df.sort_values('test_median_rrmse')
display(hist_df[['experiment_name', 'test_median_rrmse', 'train_median_rrmse', 'test_median_corr', 'is_best']])